# Week 24 (Capstone) — Safe candidate matching

The capstone notebook. Eight graded exercises that walk through the full pipeline:

    data → IQL pretrain → PPO-Lagrangian fine-tune → fairness audit

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as ref

## Exercise 1 ★ — Dataset synthesis

Generate the synthetic candidate-job-interaction dataset and confirm the shapes.

> **Hint**: use `ref.exercise_1`.

In [ ]:
def exercise_1(seed=0):
    # YOUR CODE HERE
    return ref.exercise_1(seed=seed)

In [ ]:
out = exercise_1(seed=0)
print(out)
assert out['n_jobs'] == 20 and out['n_candidates'] == 200

## Exercise 2 ★ — Bias in the logged data

Measure the demographic-parity gap of the *logger* (not the learned policy). It should be clearly above zero — that's the bias we're correcting.

> **Hint**: compute `|P(a=1|g=0) - P(a=1|g=1)|` over the logged interactions.

In [ ]:
def exercise_2(seed=0):
    return ref.exercise_2(seed=seed)

In [ ]:
bias = exercise_2(seed=0)
print('logger DP gap =', bias)
assert bias > 0.0

## Exercise 3 ★★ — IQL pretrain

Fit the offline-RL model and inspect the Q / V / pi loss curves.

> **Hint**: use `ref.exercise_3`; plot `history['q_loss']`, `history['v_loss']`, `history['pi_loss']`.

In [ ]:
def exercise_3(seed=0):
    return ref.exercise_3(seed=seed)

In [ ]:
out = exercise_3(seed=0)
h = out['history']
fig, ax = plt.subplots(figsize=(6.5, 3.5))
for k, c in zip(['q_loss','v_loss','pi_loss'], ['blue','teal','orange']):
    ax.plot(h[k], color=RL_PALETTE[c], label=k)
ax.legend(); ax.set_xlabel('Epoch'); plt.show()

## Exercise 4 ★★ — SNIPS off-policy evaluation

Estimate the value of a uniformly random target policy by SNIPS on the logged data.

> **Hint**: the target picks each action with prob 0.5; the importance weights are 0.5/propensity.

In [ ]:
def exercise_4(seed=0):
    return ref.exercise_4(seed=seed)

In [ ]:
v = exercise_4(seed=0)
print('SNIPS value of uniform-random target =', v)

## Exercise 5 ★★ — PPO-Lagrangian fine-tune

Fine-tune the IQL-pretrained policy online with a fairness cost. Report the parity trajectory.

> **Hint**: `ref.exercise_5(seed=0)` returns a history dict with keys `return`, `cost`, `lambda`, `parity`.

In [ ]:
def exercise_5(seed=0):
    return ref.exercise_5(seed=seed)

In [ ]:
out = exercise_5(seed=0)
h = out['history']
fig, axes = plt.subplots(1,3, figsize=(13,3.5))
axes[0].plot(h['return'], color=RL_PALETTE['blue']); axes[0].set_title('Return')
axes[1].plot(h['parity'], color=RL_PALETTE['red']); axes[1].axhline(0.08, color=RL_PALETTE['gray'], linestyle=':'); axes[1].set_title('Parity gap')
axes[2].plot(h['lambda'], color=RL_PALETTE['orange']); axes[2].set_title(r'$\lambda$')
plt.tight_layout(); plt.show()
print('final parity:', out['final_parity'])

## Exercise 6 ★★★ — Meaningful objective: end-to-end audit

Run the full pipeline and inspect the audit dict BEFORE and AFTER Lagrangian fine-tune. Discuss which metrics improved.

This is the bridge to ATS deployment: in production, this same dict goes onto the audit dashboard for every retrain.

In [ ]:
def exercise_6(seed=0):
    return ref.exercise_6(seed=seed)

In [ ]:
out = exercise_6(seed=0)
print('BEFORE')
for k in ('demographic_parity_gap','equal_opportunity_gap','snips_value','mean_rec_rate'):
    print(f'  {k:30s} = {out["audit_before"][k]:.4f}')
print('AFTER')
for k in ('demographic_parity_gap','equal_opportunity_gap','snips_value','mean_rec_rate'):
    print(f'  {k:30s} = {out["audit_after"][k]:.4f}')

## Exercise 7 ★★★ — Counterfactual red-team

Flip the protected attribute bit in the context and measure how often the top-1 decision changes. Below 5% disagreement is a reasonable bar.

> **Hint**: this is the *counterfactual fairness* probe; it catches attribute leakage even when DP/EO look fine.

In [ ]:
def exercise_7(seed=0):
    return ref.exercise_7(seed=seed)

In [ ]:
g = exercise_7(seed=0)
print('counterfactual group-flip disagreement =', g)

## Exercise 8 ★★★ — Open-ended: Pareto frontier

Sweep the cost limit `d` in {0.0, 0.05, 0.10, 0.20} and trace the (parity, value) frontier. Comment on the operating point you would defend to compliance.

> **Hint**: this is the fairness-value Pareto frontier; pick the knee.

In [ ]:
def exercise_8(seed=0):
    return ref.exercise_8(seed=seed)

In [ ]:
pts = exercise_8(seed=0)
fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.plot([p['parity_gap'] for p in pts], [p['snips_value'] for p in pts], 'o-', color=RL_PALETTE['blue'])
for p in pts:
    ax.annotate(f"d={p['cost_limit']}", (p['parity_gap'], p['snips_value']), textcoords='offset points', xytext=(5,5), fontsize=8)
ax.set_xlabel('Parity gap (lower fairer)'); ax.set_ylabel('SNIPS value (higher better)'); ax.set_title('Pareto frontier')
plt.show()
print(pts)

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

_A consolidated cheat-sheet — read only after you've attempted each exercise. Stronger than the inline hints above the scaffolds, weaker than the full solution. Worked answers are in `project/solutions.py` in the same folder (`week-24-capstone-safe-matching`).
_

### Exercise 1 ★ — Dataset synthesis

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 2 ★ — Bias in the logged data

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 3 ★★ — IQL pretrain

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 4 ★★ — SNIPS off-policy evaluation

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 5 ★★ — PPO-Lagrangian fine-tune

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 6 ★★★ — Meaningful objective: end-to-end audit

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 7 ★★★ — Counterfactual red-team

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.

### Exercise 8 ★★★ — Open-ended: Pareto frontier

- **Key relation:** look up the equation in the corresponding `notes.pdf` chapter "Mathematical formulation". The notebook prompt names the function — find that function in `project/solutions.py` and skim its docstring.
- **Expected output shape:** scalar for losses / regret, `(n_steps,)` for training curves, `(nS,)` for value functions, `(nS, nA)` for Q-tables, dicts for hyperparameter sweeps.
- **Sanity checks (in order, cheapest first):**
  1. Run on the smallest fixture first (`n_steps=10`, `n_episodes=5`).
  2. Compare against the random-policy baseline you already have.
  3. Check finite values (no `nan` / `inf` after the first step).
  4. Plot the learning curve — if it's flat, recheck the step direction (gradient sign).
- **Common pitfalls for this exercise type:**
  - Off-by-one in advantage estimation (GAE / TD): always test on a deterministic toy first.
  - Forgetting `torch.no_grad()` around target-net targets — silent inflation of the loss.
  - Updating the target / behavior policy with the same gradient — double counting.
  - Using `.detach()` on the wrong tensor in DPO / PPO — kills the gradient.
  - In off-policy / OPE: importance weights must use the behavior policy at the time the action was logged (not the current policy).
- **Worked answer:** see the matching function in `project/solutions.py` — its body shows the canonical implementation; read it only after a serious attempt.


---
_End of week. Mark this notebook in your tracker only after every exercise's hidden test cell passes._
